In [178]:
import requests
from operator import itemgetter
import pandas as pd
import matplotlib.pyplot as plt
import os

In [ ]:
users_url = 'https://jsonplaceholder.typicode.com/users'

In [ ]:
# Retrieve customer data with Requests
def get_customers(url):
    
    try:
        res = requests.get(url, timeout=3)
        res.raise_for_status()
        return res.json()
    except requests.exceptions.RequestException:
        print('Network error: Unable to connect. Try again later')
        return None


In [ ]:
get_customers(users_url)


In [ ]:
# Explore nested dictionaries and lists
def explore(record, indent=0):
    prefix = ' ' * indent
    if isinstance(record, dict):
        for key, value in record.items():
            print(f'{prefix}{key}: {type(value).__name__}')

            if isinstance(value, dict):
                explore(value, indent + 4)



In [ ]:
cust1 = {'id': 1,
  'name': 'Leanne Graham',
  'username': 'Bret',
  'email': 'Sincere@april.biz',
  'address': {'street': 'Kulas Light',
   'suite': 'Apt. 556',
   'city': 'Gwenborough',
   'zipcode': '92998-3874',
   'geo': {'lat': '-37.3159', 'lng': '81.1496'}},
  'phone': '1-770-736-8031 x56442',
  'website': 'hildegard.org',
  'company': {'name': 'Romaguera-Crona',
   'catchPhrase': 'Multi-layered client-server neural-net',
   'bs': 'harness real-time e-markets'
   }}

explore(cust1)

In [73]:
# Extract customer attributes
def extract_attributes(customers):
    extracted = []

    for customer in customers:
        email = customer.get('email', '')
        address = customer.get('address', {})
        geo = address.get('geo', {})
        company = customer.get('company', {})


        record = {
            'id': customer.get('id'),
            'name': customer.get('name'),
            'username': customer.get('username'),
            'email': email,
            'email_domain': email.split('@')[-1],
            'phone': customer.get('phone'),
            'city': address['city'],
            'zipcode': address['zipcode'],
            'latitude': float(geo['lat']) if geo.get('lat') is not None else None,
            'longitude': float(geo['lng']) if geo.get('lng') is not None else None,
            'company': company['name'],
            'website': customer.get('website')
        }
        
        extracted.append(record)
    return extracted


In [ ]:
extract_attributes(get_customers(users_url))

In [82]:
# Use sets to find unique cities
def unique_cities(customers):
    unique_cities = []
    for customer in customers:
        unique_cities.append(customer['city'])
    return set(unique_cities)


In [91]:
cities = unique_cities(extract_attributes(get_customers(users_url)))
print(f'Number of Cities: {len(cities)}')
print(f'Cities: {cities}\n')

Number of Cities: 10
Cities: {'Lebsackbury', 'Bartholomebury', 'South Elvis', 'McKenziehaven', 'Roscoeview', 'Howemouth', 'Gwenborough', 'Wisokyburgh', 'Aliyaview', 'South Christy'}



In [139]:
# Count customers by city with loops and dict comprehensions
def city_counts(customers):
    customer_count = {}
    for customer in customers:
        city = customer.get('address', {}).get('city')

        if city:
            customer_count[city] = customer_count.get(city, 0) + 1

    sorted_cities = dict(sorted(customer_count.items(), key=itemgetter(1)))

    return sorted_cities



In [140]:


city_counts(get_customers(users_url))

{'Gwenborough': 1,
 'Wisokyburgh': 1,
 'McKenziehaven': 1,
 'South Elvis': 1,
 'Roscoeview': 1,
 'South Christy': 1,
 'Howemouth': 1,
 'Aliyaview': 1,
 'Bartholomebury': 1,
 'Lebsackbury': 1}

In [ ]:
def email_domain_counts(customers):
    customer_count_by_domain = {}

    for customer in customers:
        email_domain = customer.get('email', '').split('@')[-1]
        
        if email_domain:
            customer_count_by_domain[email_domain] = customer_count_by_domain.get(email_domain, 0) + 1 

    sorted_domains = dict(sorted(customer_count_by_domain.items(), key=itemgetter(1)))
    
    return sorted_domains

In [138]:
    email_domain_counts(get_customers(users_url))

{'april.biz': 1,
 'melissa.tv': 1,
 'yesenia.net': 1,
 'kory.org': 1,
 'annie.ca': 1,
 'jasper.info': 1,
 'billy.biz': 1,
 'rosamond.me': 1,
 'dana.io': 1,
 'karina.biz': 1}

In [144]:
def to_dataframe(rows):
    return pd.DataFrame(rows)

In [ ]:
customer_df = to_dataframe(get_customers(users_url))
# customer_df.groupby('email')
email_domain = customer_df['email'].str.split('@').str[-1]
customer_df['domain'] = customer_df['email'].str.split('@').str[-1]

# email_domain_counts = email_domains.value_counts()

email_domain_counts= customer_df.groupby('domain').size().reset_index(name='count')
# print(email_domain_counts)



        domain  count
0     annie.ca      1
1    april.biz      1
2    billy.biz      1
3      dana.io      1
4  jasper.info      1
5   karina.biz      1
6     kory.org      1
7   melissa.tv      1
8  rosamond.me      1
9  yesenia.net      1


In [166]:
def check_basics(df):
    print('#'*20, 'Shape', '#'*20)
    print(df.shape)
    print('#'*20, 'Data Types', '#'*20)
    print(df.dtypes)

In [167]:
check_basics(customer_df)

#################### Shape ####################
(10, 9)
#################### Data Types ####################
id           int64
name           str
username       str
email          str
address     object
phone          str
website        str
company     object
domain      object
dtype: object


In [ ]:
# Export the results to CSV

cities_df = pd.Series(city_counts(get_customers(users_url)), name='Count')

customer_df.to_csv('output/customers.csv', index=False)
cities_df.to_csv('output/city_counts.csv', index_label='City')

In [202]:
def plot_distribution(df, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    plt.figure(figsize=(10,12))
    df.plot(kind='bar')
    plt.title('Customer distribution')
    plt.xlabel('City')
    plt.ylabel('Customer Count')
    plt.xticks(rotation=45)
    plt.savefig(path, dpi=300)
    plt.close()



In [203]:
plot_distribution(cities_df, 'output/customer_distribution.png')